In [41]:
#%pip install pandas numpy faker
#%pip install faker faker-vehicle


In [42]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import random
import string
from faker_vehicle import VehicleProvider

In [43]:
fake = Faker('en_US')
random.seed(42)
Faker.seed(42)

num_records = 200000


In [ ]:
fake.address()

In [45]:
def introduce_noise(data, error_rate=0.05, null_rate=0.0):
    noisy_data = []
    for val in data:
        rand = random.random()
        if rand < null_rate:
            noisy_data.append(None)
        elif rand < null_rate + error_rate:
            noisy_data.append("ERROR")
        else:
            noisy_data.append(val)
    return noisy_data

def introduce_date_noise(dates, error_rate=0.05):
    noisy_dates = []
    for date in dates:
        rand = random.random()
        if rand < error_rate / 3:
            noisy_dates.append(date.strftime('%d/%m/%y'))
        elif rand < 2 * error_rate / 3:
            noisy_dates.append(date.strftime('%m/%d/%y'))
        # elif rand < error_rate:
        #     noisy_dates.append(date.strftime('%y/%m/%d'))
        # else:
        #     noisy_dates.append(date.strftime('%Y-%m-%d'))
        else:
            noisy_dates.append(date.strftime('%b/%y'))
    return noisy_dates

def generate_random_part_numbers(prefix, count):
    part_numbers = set()
    while len(part_numbers) < count:
        suffix = ''.join(random.choices(string.digits, k=3))
        part_numbers.add(f"{prefix}{suffix}")
    return list(part_numbers)

In [46]:
# --- Part Catalog ---
part_catalog = {
    "Spark Plug": generate_random_part_numbers("SP6",22),
    "Fuel Pump": generate_random_part_numbers("FP7", 20),
    "Oil Filter": generate_random_part_numbers("OF8", 20),
    "Air Filter": generate_random_part_numbers("AF9", 20),
    "Ignition Coil": generate_random_part_numbers("IC0", 18),
    "Brake Rotor": generate_random_part_numbers("BR5", 12),
    "Wiper Blade": generate_random_part_numbers("WB3", 15),
    "Tailgate Lift Support": generate_random_part_numbers("TLS4", 10),
    "Engine Water Pump": generate_random_part_numbers("EWP1", 10),
}

In [47]:

# Generate parts and engine data
part_types = []
part_numbers = []
for _ in range(num_records):
    part = random.choice(list(part_catalog.keys()))
    part_types.append(part)
    part_numbers.append(random.choice(part_catalog[part]))

cc_cluster_1 = np.random.normal(loc=1400, scale=200, size=num_records // 2)
cc_cluster_2 = np.random.normal(loc=3500, scale=500, size=num_records // 2)
engine_cc = np.concatenate([cc_cluster_1, cc_cluster_2])
np.random.shuffle(engine_cc)
engine_cc = np.clip(engine_cc, 600, 6000)


# Base IDs
invoice_ids = [f"INV{str(i).zfill(6)}" for i in range(1, num_records + 1)]
customer_ids = [f"CUST{str(i % 10000).zfill(4)}" for i in range(1, num_records + 1)]

# Base distributions
service_costs = np.random.lognormal(mean=7.0, sigma=0.6, size=num_records)
odometers = np.random.lognormal(mean=10.0, sigma=0.5, size=num_records)
fuel_consumption = np.random.normal(loc=7.5, scale=1.5, size=num_records)
engine_temp = np.random.normal(loc=90, scale=10, size=num_records)

repair_duration = np.random.normal(loc=3, scale=1.5, size=num_records)
number_of_visits = np.random.poisson(lam=2, size=num_records)
parts_cost =  [round(random.uniform(5, 100), 2) for part in part_numbers]
labor_cost = np.random.normal(loc=10, scale=3, size=num_records)
discount_amount = np.random.normal(loc=50, scale=30, size=num_records)
insurance_coverage = np.random.uniform(low=70, high=100, size=num_records)
service_rating = np.random.normal(loc=4.2, scale=0.8, size=num_records)

# Clip ranges to make values realistic
fuel_consumption = np.clip(fuel_consumption, 2, 30)
engine_temp = np.clip(engine_temp, 50, 200)
repair_duration = np.clip(repair_duration, 0.5, 15)
# labor_cost = np.clip(labor_cost, 50, 1000)
discount_amount = np.clip(discount_amount, 0, 500)
service_rating = np.clip(service_rating, 1, 5)



# Initialize Faker
fake = Faker()
fake.add_provider(VehicleProvider)

# List of sample customers
customers = ["NAPA", "AutoZone", "O'Reilly", "Advance Auto Parts", "CarQuest", "Summit Racing", "RockAuto","Amazon", "Walmart"]


make = []
model = []
year = []
customer = []

for _ in range(num_records):
    vehicle = fake.vehicle_object()
    make.append(vehicle['Make'])
    model.append(vehicle['Model'])
    year.append(vehicle['Year'])
    customer.append(random.choice(customers))  # Random customer

In [48]:
vehicle = fake.vehicle_object()
print(vehicle)


{'Year': 2018, 'Make': 'Alfa Romeo', 'Model': 'Stelvio', 'Category': 'SUV'}


In [49]:
# Data Dictionary
data = {
    "Invoice_ID": invoice_ids,
    "Customer_ID": customer_ids,
    "Customer_Name": [fake.name() for _ in range(num_records)],
    # "Country": ["USA"] * num_records,
    "State": [fake.state() for _ in range(num_records)],

    "Vehicle_Year": year,
    "Vehilce_Make": make,
    "Vehicle_Model": model,
    "Customer": customer,
    "Purchase_Date": introduce_date_noise([fake.date_between(start_date='-5y', end_date='today') for _ in range(num_records)]),
    "Service_Date": introduce_date_noise([fake.date_between(start_date='-2y', end_date='today') for _ in range(num_records)]),
    "Warranty_Expiry": introduce_date_noise([fake.date_between(start_date='today', end_date='+5y') for _ in range(num_records)]),

    # "Service_Cost_USD": introduce_noise([round(val, 2) for val in service_costs]),
    "Odometer_km": introduce_noise([int(val) for val in odometers]),
    "Fuel_Consumption_L_per_100km": introduce_noise([round(val, 2) for val in fuel_consumption]),
    "Engine_CC": introduce_noise([int(round(cc / 100) * 100) for cc in engine_cc]),
    "Engine_Temp_C": introduce_noise([round(val, 1) for val in engine_temp]),
    "Feedback_Score": introduce_noise([random.choice([1, 2, 3, 4, 5]) for _ in range(num_records)]),
    "Payment_Method": [random.choice(["Credit Card", "Cash", "Online Transfer", "Cheque"]) for _ in range(num_records)],
    "Part_Type": part_types,
    "Part_Number": part_numbers,

    "Repair_Duration_Hours": introduce_noise([round(val, 2) for val in repair_duration]),
    # "Number_of_Visits": introduce_noise([int(val) for val in number_of_visits]),
    #"Parts_Cost_USD": introduce_noise([round(val, 2) for val in parts_cost]),
    "Labor_Cost_USD": introduce_noise([round(val, 2) for val in labor_cost]),
    "Discount_Amount_USD": introduce_noise([round(val, 2) for val in discount_amount]),
    "Insurance_Coverage_Percent": introduce_noise([round(val, 2) for val in insurance_coverage]),
    "Service_Rating": introduce_noise([round(val, 2) for val in service_rating]),
}

In [ ]:
df = pd.DataFrame(data)
df

In [51]:
unique_parts = df['Part_Number'].unique()
cost_mapping = {part: round(random.uniform(10, 100), 2) for part in unique_parts}
df['Parts_Cost_USD'] = df['Part_Number'].map(cost_mapping)

In [ ]:
df

In [ ]:
df[['Invoice_ID', 'Customer_ID', 'Customer_Name', 'State', 'Customer','Vehicle_Year','Vehilce_Make', 'Vehicle_Model','Engine_CC', 'Purchase_Date','Warranty_Expiry', 'Odometer_km','Fuel_Consumption_L_per_100km',  'Engine_Temp_C', 'Service_Date','Part_Type', 'Part_Number','Repair_Duration_Hours', 'Parts_Cost_USD', 'Labor_Cost_USD', 'Discount_Amount_USD','Insurance_Coverage_Percent', 'Service_Rating','Feedback_Score', 'Payment_Method']]

In [54]:
CustomerNames=df.Customer.unique()

In [ ]:
for i in range (len(CustomerNames)):
    df[df['Customer']==CustomerNames[i]].to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Warranty\{CustomerNames[i]}_WarrantyData.csv", index=False)
    print(f"✅ Dataset saved as '{CustomerNames[i]}_WarrantyData.csv'")


In [56]:
sales_records = 200000

In [57]:
customer_sales = []
part_types = []
part_numbers = []
for _ in range(sales_records):
    part = random.choice(list(part_catalog.keys()))
    part_types.append(part)
    part_numbers.append(random.choice(part_catalog[part]))
    customer_sales.append(random.choice(customers))

In [58]:
sales_data={
    "Part_Type": part_types,
    "Part_Number": part_numbers,
    "Customer": customer_sales,
    "Purchase_Date":[fake.date_between(start_date='-5y', end_date='today') for _ in range(sales_records)],
    "Units_Sold": np.random.poisson(lam=5, size=sales_records),
}

In [ ]:
df_sales=pd.DataFrame(sales_data)
df_sales

In [60]:
df_sales['Purchase_Date'] = pd.to_datetime(df_sales['Purchase_Date'])
df_sales['Month'] = df_sales['Purchase_Date'].dt.strftime('%Y-%m') # Format as 'YYYY-MM' for consistent ordering

In [61]:
pivot_df = pd.pivot_table(df_sales, 
                            values='Units_Sold', 
                            index=["Part_Type",'Part_Number',"Customer"], 
                            columns='Month', 
                            aggfunc='sum', 
                            fill_value=0)

In [ ]:
pivot_df

In [ ]:
# unique_parts = df_sales['Part_Number'].unique()
# cost_mapping = {part: round(random.uniform(10, 100), 2) for part in unique_parts}
# df_sales['Parts_Cost_USD'] = df_sales['Part_Number'].map(cost_mapping)

In [63]:
pivot_df.to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\Sales_data.csv", index=True)

In [64]:
df_sales_updated=pd.read_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\Sales_data.csv")

In [65]:
PartTypes= df_sales['Part_Type'].unique().tolist()

In [ ]:
for i in range (len(PartTypes)):
    df_sales_updated[df_sales_updated['Part_Type']==PartTypes[i]].to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\{PartTypes[i]}_SalesData.csv", index=False)
    print(f"✅ Dataset saved as '{PartTypes[i]}_WarrantyData.csv'")
